Cell 1: Imports and Global Constants


In [16]:
import cv2
import os
import numpy as np
import tkinter as tk
from tkinter import filedialog, messagebox
from PIL import Image, ImageTk
from sklearn import linear_model
from sklearn.model_selection import train_test_split

# Global Constants
IMG_SIZE = (16, 16)  # Standard size for training and prediction
TRAIN_DIRS = [str(i) for i in range(1, 10)]  # Folders '1' to '9'


Cell 2: The Core Machine Learning Logic


In [17]:
class DigitTrainer:
    def __init__(self, train_dirs, img_size):
        self.train_dirs = train_dirs
        self.img_size = img_size
        self.model = linear_model.LogisticRegression(max_iter=1000)

    def train(self):
        """
        Loads images from directories, preprocesses them, 
        and trains the Logistic Regression model.
        """
        X = []
        y = []

        for label_dir in self.train_dirs:
            if not os.path.exists(label_dir):
                print(f"Warning: Directory {label_dir} not found. Skipping.")
                continue
                
            label = int(label_dir)
            files = os.listdir(label_dir)
            
            for file in files:
                file_path = os.path.join(label_dir, file)
                # Using numpy to read image to handle special characters in paths
                try:
                    img_array = np.fromfile(file_path, np.uint8)
                    img = cv2.imdecode(img_array, cv2.IMREAD_GRAYSCALE)
                    
                    if img is None:
                        continue

                    # Preprocessing: Resize and Flatten
                    img_resized = cv2.resize(img, self.img_size)
                    img_flattened = img_resized.flatten() / 255.0
                    
                    X.append(img_flattened)
                    y.append(label)
                except Exception as e:
                    print(f"Error loading {file_path}: {e}")

        if len(X) == 0:
            return False

        X = np.array(X)
        y = np.array(y)

        # Train the model
        self.model.fit(X, y)
        print(f"Model training complete. Processed {len(y)} samples.")
        return True

    def predict(self, flattened_img):
        """Predicts a single digit using the trained model."""
        return self.model.predict([flattened_img])[0]


Cell 3: Plate Processing Logic


In [18]:
def process_plate_logic(image_path, trainer):
    """
    Segments the plate using Projection Profile, 
    draws red rectangles around detected digits, 
    and predicts them.
    """
    # 1. Load Image safely
    try:
        img_array = np.fromfile(image_path, np.uint8)
        img = cv2.imdecode(img_array, cv2.IMREAD_COLOR)
    except Exception as e:
        return None, f"Error: {e}"

    if img is None:
        return None, "Image not found"

    # 2. Preprocessing
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    # Use Otsu's Binarization to separate digit from background
    # We use THRESH_BINARY_INV so digits become white (255) and background black (0)
    _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    
    output_img = img.copy()
    
    # 3. Segmentation via Projection Profile (Column Sums)
    column_sums = np.sum(thresh, axis=0)
    
    detected_digits = []
    start_col = -1
    
    # Threshold for column sum to define what is a 'digit' vs 'background'
    # Adjust this value if detection fails
    COLUMN_THRESHOLD = 20 

    for col in range(thresh.shape[1]):
        if column_sums[col] > COLUMN_THRESHOLD:
            if start_col == -1:
                start_col = col
        else:
            if start_col != -1:
                end_col = col
                width = end_col - start_col
                
                # Filter out noise (too narrow segments)
                if width > 5:
                    # a. Crop the digit
                    digit_segment = thresh[:, start_col:end_col]
                    
                    # b. Resize and flatten for the model
                    digit_resized = cv2.resize(digit_segment, IMG_SIZE)
                    digit_flat = digit_resized.flatten() / 255.0
                    
                    # c. Predict
                    try:
                        prediction = trainer.predict(digit_flat)
                        detected_digits.append(str(prediction))
                        
                        # d. Draw Red Rectangle around the digit
                        # Syntax: cv2.rectangle(img, start_point, end_point, color, thickness)
                        cv2.rectangle(output_img, (start_col, 0), (end_col, img.shape[0]), (0, 0, 255), 2)
                    except:
                        pass

                start_col = -1
                
    result_str = "".join(detected_digits)
    if not result_str:
        result_str = "No digits detected"
        
    return output_img, result_str


Cell 4: Tkinter GUI Implementation


In [19]:
class LicensePlateApp:
    def __init__(self, root, trainer):
        self.root = root
        self.trainer = trainer
        self.root.title("AI License Plate Digit Recognizer")
        self.root.geometry("800x600")
        
        self.setup_ui()

    def setup_ui(self):
        # Header
        self.label_title = tk.Label(self.root, text="License Plate Recognition System", font=("Arial", 18, "bold"))
        self.label_title.pack(pady=10)

        # Canvas for Image Display
        self.canvas = tk.Canvas(self.root, width=600, height=400, bg="gray")
        self.canvas.pack(pady=10)

        # Result Label
        self.label_result = tk.Label(self.root, text="Result: ---", font=("Arial", 16))
        self.label_result.pack(pady=5)

        # Buttons
        self.btn_select = tk.Button(self.root, text="Select Plate Image", command=self.load_image, width=20, height=2)
        self.btn_select.pack(pady=10)

    def load_image(self):
        file_path = filedialog.askopenfilename(filetypes=[("Image files", "*.png *.jpg *.jpeg")])
        if not file_path:
            return

        # Process the image
        processed_img, result_text = process_plate_logic(file_path, self.trainer)

        if processed_img is not None:
            # Convert OpenCV image to PIL for Tkinter
            rgb_img = cv2.cvtColor(processed_img, cv2.COLOR_BGR2RGB)
            pil_img = Image.fromarray(rgb_img)
            
            # Resize for display in Canvas
            pil_img.thumbnail((600, 400))
            self.tk_img = ImageTk.PhotoImage(pil_img)
            
            self.canvas.create_image(300, 200, image=self.tk_img)
            self.label_result.config(text=f"Result: {result_text}", fg="blue")
        else:
            messagebox.showerror("Error", f"Failed to process image: {result_text}")


Cell 5: Main Execution


In [20]:
if __name__ == "__main__":
    # 1. Initialize Trainer
    print("Initializing and Training Model...")
    trainer = DigitTrainer(TRAIN_DIRS, IMG_SIZE)
    
    # 2. Run Training
    success = trainer.train()
    
    if not success:
        print("Error: Training failed. Check if folders 1-9 exist and contain images.")
    else:
        # 3. Launch GUI
        root = tk.Tk()
        app = LicensePlateApp(root, trainer)
        root.mainloop()


Initializing and Training Model...
Model training complete. Processed 3101 samples.
